In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of dscim_refactor.pipeline.steps.save failed: Traceback (most recent call last):
  File "/Users/sebastiancadavidsanchez/miniconda3/envs/dscim_stable_py311/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 283, in check
    superreload(m, reload, self.old_objects)
  File "/Users/sebastiancadavidsanchez/miniconda3/envs/dscim_stable_py311/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 483, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/Users/sebastiancadavidsanchez/miniconda3/envs/dscim_stable_py311/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/Users/sebastiancadavidsanchez/Documents/Github/cil/dscim-testing/dscim_refactor/pipeline/steps/sa

This notebook demonstrates how to construct, run, and inspect a modular DSCIM pipeline for reducing climate damages using a simplified configuration for testing and validation.

The pipeline is composed of modular steps:

1. **`LoadSectorDataStep`** – Loads sector-specific damage and socioeconomic input data.
2. **`ApplyReductionStep`** – Applies a damage reduction recipe (e.g., `adding_up`, `risk_aversion`).
3. **`SaveReducedDataStep`** – Saves the reduced dataset to a Zarr file, either using the default config path or a custom output location.

## Key Features

- Interactive pipeline visualization before and after execution, with detailed metadata and output dimensions per step.
- Optional configuration of custom output locations to replace the default save path.
- Node-level execution status tracking to highlight which steps succeeded or failed.
- Designed for verifying that the refactored pipeline produces equivalent results to the original `reduce_damages` function.

> This notebook serves as a foundation for interactive debugging and comparison against the legacy implementation.


In [17]:
# path to run example
import sys
from pathlib import Path
# -- Handle paths
current_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path().resolve()
project_root = current_dir.parents[1]
sys.path.insert(0, str(project_root))


# -- Import pipeline components
from dscim_refactor.pipeline.pipeline import PipelineStep
from dscim_refactor.pipeline.pipeline import DscimPipeline
from dscim_refactor.pipeline.steps.load_sector import LoadSectorDataStep
from dscim_refactor.pipeline.steps.transform import ApplyReductionStep
from dscim_refactor.pipeline.steps.save import SaveReducedDataStep
from dscim_refactor.config.config_loader import ConfigLoader
from dscim_refactor.utils.pathing import get_project_root
from dscim_refactor.utils.logger import setup_logger  
from IPython.display import display

from itertools import product
import logging

def run_pipeline(pipeline):
    """Wrapper to hide prints when saving zarr files"""
    # print("=== Running pipeline ===")
    pipeline.compute()
    # print("=== Pipeline execution completed ===")

# -- Set up logger (modify `level` for less prints)
logger = setup_logger(__name__, level=logging.ERROR) # DEBUG, INFO, WARNING, ERROR, CRITICAL

# -- Find project root and load config
project_root = get_project_root()
config_path = project_root / "configs" / "dummy_config.yaml"
config = ConfigLoader(config_path, base_path=project_root)

logger.info("Configuration successfully loaded.")

In [18]:
# Load configuration
dummy_config_path = "../../configs/dummy_config.yaml"
config = ConfigLoader(dummy_config_path, base_path=get_project_root())


# Define sectors and reductions
sectors = {
    "coastal": "dummy_coastal_sector",
    "not_coastl": "dummy_not_coastl_sector"
}

# Which sectors to run
sectors.update(
    indiv_sectors=[sectors["not_coastl"]]
)

reductions = ["cc", "no_cc"]
recipe_discs = list(
    product(
        ["adding_up", "risk_aversion"],
        [None]  # Simplified: using only one discount for dummy run
    )
)

# Set eta values manually
eta_rhos = [[2.0, 0.0001]]

# Assume a bottom coding value passed manually for now
BOTTOM_CODING_GDPPC = 39.39265060424805


## Pipeline Configuration and Execution

This section selects a single configuration (sector, reduction, recipe) and demonstrates two options for saving the reduced output:

- **Option 1 (default):** Uses the output path defined in the configuration file (`config.paths["reduced_damages_library"]`).
- **Option 2 (custom):** Allows overriding the base output directory while preserving the standard filename pattern.

Only one save option should be used when building the pipeline. The pipeline is then visualized and executed step-by-step to verify structure and outputs.

In [19]:
# ------------------------
# Select one configuration
# ------------------------

sector = sectors["indiv_sectors"][0]
reduction = reductions[0]
recipe = "adding_up"

# ------------------------------------------------
# Option 1: Use default save base path (from config)
# ------------------------------------------------
# Saves to: config.paths["reduced_damages_library"] / <sector>_<recipe>_<reduction>.zarr

default_save_step = SaveReducedDataStep(
    name="SaveReduced",
    sector=sector,
    config=config,
    recipe=recipe,
    reduction=reduction
)

# ------------------------------------------------
# Option 2: Use custom base directory, keep file name pattern
# ------------------------------------------------
# You just change the root folder, not the full filename.
# Result: ../../refactor_dummy_data/dummy_not_coastl_adding_up_cc.zarr

custom_base_path = Path("../../refactor_dummy_data")

custom_save_step = SaveReducedDataStep(
    name="SaveReduced",
    sector=sector,
    config=config,
    recipe=recipe,
    reduction=reduction,
    base_output_path=custom_base_path  # this replaces config.paths["reduced_damages_library"]
)

# --------------------
# Build and run pipeline
# --------------------

pipeline = DscimPipeline([
    LoadSectorDataStep(name="LoadSector", sector=sector, config=config),
    ApplyReductionStep(
        name="ApplyReduction",
        recipe=recipe,
        reduction=reduction,
        bottom_coding_gdppc=BOTTOM_CODING_GDPPC
    ),
    # Choose one:
    # default_save_step,
    custom_save_step
])

print("==== PIPELINE BEFORE COMPUTE ====")
display(pipeline.visualize_interactive())

run_pipeline(pipeline)

==== PIPELINE BEFORE COMPUTE ====


[02:09:40] INFO     [SaveReduced] Writing reduced data to:                                                         
                    ../../refactor_dummy_data/dummy_not_coastl_sector/adding_up_cc.zarr

[02:09:40] INFO     [SaveReduced] Completed write to:                                                              
                    ../../refactor_dummy_data/dummy_not_coastl_sector/adding_up_cc.zarr

In [20]:
# Force display after compute
print("==== PIPELINE AFTER COMPUTE ====")
display(pipeline.visualize_interactive())

==== PIPELINE AFTER COMPUTE ====


## Batch Execution with Optional Custom Output Path

This section runs the full set of pipeline combinations across sectors, reductions, and recipes.  
It introduces a toggle (`use_custom_path`) that controls whether to use the default output directory (from the config file) or override it with a custom base path (e.g., `../../refactor_dummy_data`).

The helper function `make_save_step()` dynamically constructs the final save step with the appropriate path logic, ensuring consistency and flexibility across all pipeline runs.


In [12]:
# Toggle this to switch between default and custom output path
use_custom_path = True
custom_base_path = Path("../../refactor_dummy_data") if use_custom_path else None

def make_save_step(sector, recipe, reduction, eta=None):
    kwargs = dict(
        name="SaveReduced",
        sector=sector,
        config=config,
        recipe=recipe,
        reduction=reduction,
        eta=eta,
    )
    if custom_base_path:
        kwargs["base_output_path"] = custom_base_path
    return SaveReducedDataStep(**kwargs)

######################
# Run Pipeline
######################

for sector, reduction in product(sectors["indiv_sectors"], reductions):
    for recipe, _ in recipe_discs:

        if recipe == "adding_up":
            pipeline = DscimPipeline([
                LoadSectorDataStep(name="LoadSector", sector=sector, config=config),
                ApplyReductionStep(name="ApplyReduction", recipe=recipe, reduction=reduction, bottom_coding_gdppc=BOTTOM_CODING_GDPPC),
                make_save_step(sector, recipe, reduction)
            ])
            run_pipeline(pipeline)

        elif recipe == "risk_aversion":
            for eta, _ in eta_rhos:
                pipeline = DscimPipeline([
                    LoadSectorDataStep(name="LoadSector", sector=sector, config=config),
                    ApplyReductionStep(name="ApplyReduction", recipe=recipe, reduction=reduction, bottom_coding_gdppc=BOTTOM_CODING_GDPPC, eta=eta),
                    make_save_step(sector, recipe, reduction, eta)
                ])
                run_pipeline(pipeline)

print("Dummy damage reduction pipeline completed.")


[01:51:36] INFO     [SaveReduced] Writing reduced data to:                                                         
                    ../../refactor_dummy_data/dummy_not_coastl_sector/adding_up_cc.zarr

[01:51:36] INFO     [SaveReduced] Completed write to:                                                              
                    ../../refactor_dummy_data/dummy_not_coastl_sector/adding_up_cc.zarr

[01:51:36] INFO     [SaveReduced] Writing reduced data to:                                                         
                    ../../refactor_dummy_data/dummy_not_coastl_sector/risk_aversion_cc_eta2.0.zarr

[01:51:36] INFO     [SaveReduced] Completed write to:                                                              
                    ../../refactor_dummy_data/dummy_not_coastl_sector/risk_aversion_cc_eta2.0.zarr

[01:51:36] INFO     [SaveReduced] Writing reduced data to:                                                         
                    ../../refactor_dummy_data/dummy_not_coastl_sector/adding_up_no_cc.zarr

[01:51:37] INFO     [SaveReduced] Completed write to:                                                              
                    ../../refactor_dummy_data/dummy_not_coastl_sector/adding_up_no_cc.zarr

[01:51:37] INFO     [SaveReduced] Writing reduced data to:                                                         
                    ../../refactor_dummy_data/dummy_not_coastl_sector/risk_aversion_no_cc_eta2.0.zarr

[01:51:37] INFO     [SaveReduced] Completed write to:                                                              
                    ../../refactor_dummy_data/dummy_not_coastl_sector/risk_aversion_no_cc_eta2.0.zarr

Dummy damage reduction pipeline completed.
